# Cruce de dos archivos de Excel — versión corta

Mismo resultado que la versión larga, en **7 bloques y ~55 líneas** de código.

| Bloque | Qué hace |
|---|---|
| 1 | Librerías y rutas |
| 2 | Las 3 funciones (único bloque largo: se escribe una vez y no se toca) |
| 3 | Ver qué archivos hay |
| 4 | Cargar los dos archivos y ver sus columnas numeradas |
| 5 | Probar cuál normalización de la llave cruza mejor |
| 6 | Cruzar |
| 7 | Revisar y exportar |

**Convención:** `A` es tu archivo base (lo conservas completo), `B` es del que traes datos.

---
## BLOQUE 1 — Librerías y rutas ✏️

`pandas` maneja las tablas y `openpyxl` lee/escribe `.xlsx` (pandas lo usa por debajo,
no hay que importarlo).

Cambia las dos rutas. La `r` antes de las comillas es para que las barras de Windows no den problema.

In [ ]:
import os
import pandas as pd

CARPETA = r"C:\Users\usuario\Escritorio\Cruces"
SALIDA = r"C:\Users\usuario\Escritorio\Cruces\resultado.xlsx"

---
## BLOQUE 2 — Las 3 funciones

Este es el único bloque largo. Se escribe una vez y no se vuelve a tocar.

**`leer`** — lee el archivo **todo como texto** (`dtype=str`). Esto es lo más importante de todo:
si pandas lee los identificadores como número, `0012345` se convierte en `12345` (pierdes los ceros)
y `900123456` se convierte en `900123456.0` (deja de cruzar). El `.replace` deshace ese `.0`.

**`ver`** — imprime las columnas **numeradas**. Esos números son los que usas en los bloques 5 y 6.

**`llave`** — deja el identificador listo para cruzar. 4 modos:

| Modo | Qué hace | Ejemplo |
|---|---|---|
| `texto` | Sin tildes, MAYÚSCULAS, un solo espacio | `" Gámma  S.A. "` → `GAMMA S.A.` |
| `nit` | Solo dígitos (quita puntos, guiones, espacios) | `900.123.456-7` → `9001234567` |
| `nit_sin_dv` | Solo dígitos **y quita el dígito de verificación** | `900.123.456-7` → `900123456` |
| `sin_ceros` | Sin espacios, MAYÚSCULAS, **sin ceros a la izquierda** | `0012345` → `12345` |

`sin_ceros` es el que resuelve los identificadores internos: sirve tanto si el archivo conservó
los ceros como si Excel se los comió, porque deja a ambos lados en la misma forma.

In [ ]:
def leer(nombre):
    """Lee un archivo de la carpeta COMPLETAMENTE como texto."""
    ruta = os.path.join(CARPETA, nombre)
    if nombre.lower().endswith(".csv"):
        df = pd.read_csv(ruta, dtype=str, sep=None, engine="python")
    else:
        df = pd.read_excel(ruta, dtype=str)
    df = df.fillna("")
    return df.apply(lambda c: c.str.replace(r"^(\d+)\.0+$", r"\1", regex=True).str.strip())


def ver(df, titulo):
    """Imprime las columnas NUMERADAS con ejemplos."""
    print(f"\n--- {titulo}: {len(df)} filas ---")
    for i, c in enumerate(df.columns):
        print(f"  [{i}] {c[:28]:28} {' | '.join(df[c].head(3))[:42]}")


def llave(serie, modo):
    """Normaliza un identificador para poder cruzarlo."""
    s = serie.str.strip().str.normalize("NFKD")
    s = s.str.encode("ascii", "ignore").str.decode("ascii").str.upper()
    if modo == "texto":
        return s.str.replace(r"\s+", " ", regex=True)
    if modo == "sin_ceros":
        return s.str.replace(r"\s+", "", regex=True).str.lstrip("0")
    s = s.str.replace(r"\D", "", regex=True)          # deja solo digitos
    if modo == "nit_sin_dv":
        return s.str[:-1]                              # quita el digito de verificacion
    return s                                           # modo "nit"

---
## BLOQUE 3 — ¿Qué archivos hay?

Lista numerada de la carpeta. Anota los dos números que te interesan.

In [ ]:
ARCHIVOS = sorted(f for f in os.listdir(CARPETA) if not f.startswith("~$"))

for i, f in enumerate(ARCHIVOS):
    print(i, f)

---
## BLOQUE 4 — Cargar los dos archivos ✏️

Pon el índice de cada archivo según la lista de arriba. Verás las columnas de ambos, numeradas.

In [ ]:
A = leer(ARCHIVOS[0])     # archivo base
B = leer(ARCHIVOS[1])     # archivo del que traes datos

ver(A, "A")
ver(B, "B")

---
## BLOQUE 5 — Probar la llave ✏️

Pon el índice de la columna llave de cada archivo. Esto prueba las 16 combinaciones de modos y te
muestra **solo las que cruzan**, de mejor a peor.

Quédate con la primera línea: esos son los dos modos que vas a usar en el bloque 6.
Si no imprime nada, ninguna combinación cruzó — revisa que sean de verdad las columnas correctas.

In [ ]:
COL_A = A.columns[0]      # <- cambia el numero
COL_B = B.columns[0]      # <- cambia el numero

MODOS = ["texto", "nit", "nit_sin_dv", "sin_ceros"]
resultados = []
for ma in MODOS:
    ka = llave(A[COL_A], ma)
    ka = ka[ka != ""]
    for mb in MODOS:
        p = ka.isin(set(llave(B[COL_B], mb))).mean()
        if p > 0:
            resultados.append((p, ma, mb))

print(f"Cruzando '{COL_A}' contra '{COL_B}':\n")
for p, ma, mb in sorted(resultados, reverse=True):
    print(f"  {p:6.1%}   MODO_A = '{ma}'   MODO_B = '{mb}'")

---
## BLOQUE 6 — Cruzar ✏️

Copia los dos modos que ganaron, elige qué columnas traer de `B` y el tipo de cruce:

- `"left"` → todas las filas de A + lo que se encuentre de B (lo más común, tipo `BUSCARV`)
- `"inner"` → solo las filas que cruzan en ambos
- `"outer"` → todo de ambos lados

Dos cosas que el código resuelve solo:
- Las llaves **vacías** de B se descartan, para que los blancos no crucen entre sí.
- Si una columna existe en ambos archivos, la de B llega con el sufijo `_der` en vez de pisar la de A.

**¿Llave compuesta?** (por ejemplo NIT + año) Pega las dos llaves con `+ "|" +`:

```python
A["_k"] = llave(A["NIT"], "nit_sin_dv") + "|" + llave(A["Anio"], "texto")
B["_k"] = llave(B["nit"], "nit")        + "|" + llave(B["periodo"], "texto")
```

In [ ]:
MODO_A = "texto"
MODO_B = "texto"
TRAER = [1, 2]            # indices de las columnas de B que quieres traer
TIPO = "left"

A["_k"] = llave(A[COL_A], MODO_A)
B["_k"] = llave(B[COL_B], MODO_B)

Bx = B[B["_k"] != ""]                                  # las llaves vacias no cruzan
repetidas = Bx["_k"].duplicated().sum()
if repetidas:
    print(f"OJO: {repetidas} llaves repetidas en B -> se van a multiplicar filas de A\n")

R = A.merge(Bx[["_k"] + [B.columns[i] for i in TRAER]],
            on="_k", how=TIPO, suffixes=("", "_der"))

print(f"A: {len(A)} filas  ->  resultado: {len(R)} filas")
R.head()

---
## BLOQUE 7 — Revisar y exportar

Antes de dar el cruce por bueno, la pregunta es siempre **"lo que no cruzó, ¿por qué no cruzó?"**.

Se genera un Excel con dos hojas: `resultado` y `sin_cruce` (las filas de A que se quedaron sin
información). Como todo se manejó como texto, los **ceros a la izquierda se conservan**.

In [ ]:
sin_cruce = A[~A["_k"].isin(set(Bx["_k"]))]
print(f"Filas de A sin cruce: {len(sin_cruce)} de {len(A)}")
print(sin_cruce.head())

with pd.ExcelWriter(SALIDA) as w:
    R.drop(columns="_k").to_excel(w, sheet_name="resultado", index=False)
    sin_cruce.drop(columns="_k").to_excel(w, sheet_name="sin_cruce", index=False)

print(f"\nGuardado: {SALIDA}")